# 🎨 Sketch ↔ Photo Matching — Siamese Network (CUFS)

This notebook trains a Siamese network for face sketch-to-photo retrieval using the CUHK Face Sketch Database (CUFS).

**Make sure to set your runtime to GPU:** `Runtime → Change runtime type → GPU (A100 or T4)`

## 0 — Setup & GPU Check

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU")

## 1 — Upload the CUFS Dataset

Upload a **zip file** containing your raw CUFS data. The expected structure inside the zip:
```
photos/        ← 188 CUHK student face photos (.jpg)
sketches/      ← 188 CUHK student sketches (.jpg)
photo/         ← (optional) AR + XM2VTS photos
sketch/        ← (optional) AR + XM2VTS sketches
```

**Option A:** Upload a zip via the file picker below.  
**Option B:** If your data is on Google Drive, use the Drive mount cell instead.

In [ ]:
# --- Option A: Upload a zip from your computer ---
import os, zipfile
from google.colab import files

DATA_ROOT = "data/raw"
os.makedirs(DATA_ROOT, exist_ok=True)

print("Upload your CUFS zip file:")
uploaded = files.upload()

for fname in uploaded:
    print(f"Extracting {fname} ...")
    with zipfile.ZipFile(fname, "r") as z:
        z.extractall(DATA_ROOT)

# Show what we got
for root, dirs, ffiles in os.walk(DATA_ROOT):
    level = root.replace(DATA_ROOT, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/  ({len(ffiles)} files)")
    if level > 1:
        continue

In [ ]:
# --- Option B: Mount Google Drive instead ---
# Uncomment and edit the path below if your dataset is on Drive.

# from google.colab import drive
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/YOUR_CUFS_FOLDER"  # <-- edit this path

### Auto-detect nested folder
Sometimes the zip extracts into a subfolder. This cell fixes that.

In [ ]:
# If the zip extracted into a subfolder, detect and adjust DATA_ROOT
import os

def find_data_root(base):
    """Walk down until we find photos/ or sketches/ directory."""
    for root, dirs, _ in os.walk(base):
        if "photos" in dirs or "sketches" in dirs:
            return root
    return base

DATA_ROOT = find_data_root(DATA_ROOT)
print(f"Using DATA_ROOT = {DATA_ROOT}")
print("Contents:", os.listdir(DATA_ROOT))

## 2 — Dataset: Discover Pairs & Split

In [ ]:
import os
import re
import json
import random
from pathlib import Path
from typing import Optional
from collections import Counter

import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


# ---------------------------------------------------------------
# Discover photo-sketch pairs
# ---------------------------------------------------------------

def discover_pairs(data_root: str) -> list[dict]:
    data_root = Path(data_root)
    pairs = []

    # --- CUHK Student subset ---
    photos_dir = data_root / "photos"
    sketches_dir = data_root / "sketches"

    if photos_dir.exists() and sketches_dir.exists():
        photo_files = sorted(
            [f for f in photos_dir.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")]
        )
        sketch_files = sorted(
            [f for f in sketches_dir.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")]
        )
        n = min(len(photo_files), len(sketch_files))
        for i in range(n):
            identity = f"cuhk_{i:04d}"
            pairs.append({
                "photo": str(photo_files[i]),
                "sketch": str(sketch_files[i]),
                "identity": identity,
                "source": "CUHK",
            })

    # --- AR subset ---
    photo_dir = data_root / "photo"
    if photo_dir.exists():
        ar_photos = sorted([
            f for f in photo_dir.iterdir()
            if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
            and (f.name.lower().startswith("m-") or f.name.lower().startswith("w-"))
        ])
        sketch_dir = data_root / "sketch"
        if sketch_dir.exists():
            ar_sketches_jpg = sorted([
                f for f in sketch_dir.iterdir()
                if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
                and (f.name.lower().startswith("m-") or f.name.lower().startswith("w-"))
            ])
            n = min(len(ar_photos), len(ar_sketches_jpg))
            for i in range(n):
                identity = f"ar_{i:04d}"
                pairs.append({
                    "photo": str(ar_photos[i]),
                    "sketch": str(ar_sketches_jpg[i]),
                    "identity": identity,
                    "source": "AR",
                })

        # --- XM2VTS subset ---
        xm2_photos = sorted([
            f for f in photo_dir.iterdir()
            if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
            and f.name.lower().startswith("f-")
        ])
        if sketch_dir.exists():
            xm2_sketches_jpg = sorted([
                f for f in sketch_dir.iterdir()
                if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
                and f.name.lower().startswith("f-")
            ])
            n = min(len(xm2_photos), len(xm2_sketches_jpg))
            for i in range(n):
                identity = f"xm2_{i:04d}"
                pairs.append({
                    "photo": str(xm2_photos[i]),
                    "sketch": str(xm2_sketches_jpg[i]),
                    "identity": identity,
                    "source": "XM2VTS",
                })

    return pairs


def split_pairs(pairs, train_ratio=0.70, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    identities = list({p["identity"] for p in pairs})
    rng.shuffle(identities)
    n = len(identities)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_ids = set(identities[:n_train])
    val_ids = set(identities[n_train : n_train + n_val])
    splits = {"train": [], "val": [], "test": []}
    for p in pairs:
        if p["identity"] in train_ids:
            splits["train"].append(p)
        elif p["identity"] in val_ids:
            splits["val"].append(p)
        else:
            splits["test"].append(p)
    return splits


# --- Discover & split ---
pairs = discover_pairs(DATA_ROOT)
print(f"Total pairs: {len(pairs)}")
source_counts = Counter(p["source"] for p in pairs)
for src, cnt in sorted(source_counts.items()):
    print(f"  {src}: {cnt}")

SEED = 42
splits = split_pairs(pairs, seed=SEED)
print(f"\nTrain: {len(splits['train'])}  Val: {len(splits['val'])}  Test: {len(splits['test'])}")

assert len(pairs) > 0, "❌ No pairs found! Check DATA_ROOT and your folder structure."

### Preview some pairs

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(6, 9))
for row in range(3):
    p = pairs[row * (len(pairs) // 3)]
    photo = Image.open(p["photo"]).convert("RGB")
    sketch = Image.open(p["sketch"]).convert("RGB")
    axes[row, 0].imshow(photo)
    axes[row, 0].set_title(f"Photo — {p['source']}")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(sketch)
    axes[row, 1].set_title(f"Sketch — {p['identity']}")
    axes[row, 1].axis("off")
plt.tight_layout()
plt.show()

## 3 — Datasets & DataLoaders

In [ ]:
IMG_SIZE = 224

def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])


class SiamesePairDataset(Dataset):
    def __init__(self, pairs, train=True, seed=42):
        self.pairs = pairs
        self.transform = get_transforms(train)
        self.rng = random.Random(seed)
        self.id_to_pairs = {}
        for p in pairs:
            self.id_to_pairs.setdefault(p["identity"], []).append(p)
        self.identities = list(self.id_to_pairs.keys())

    def __len__(self):
        return len(self.pairs)

    def _load_image(self, path):
        return Image.open(path).convert("RGB")

    def __getitem__(self, idx):
        anchor = self.pairs[idx]
        if self.rng.random() < 0.5:
            label = 1
            partner = self.rng.choice(self.id_to_pairs[anchor["identity"]])
        else:
            label = 0
            neg_id = self.rng.choice(
                [i for i in self.identities if i != anchor["identity"]]
            )
            partner = self.rng.choice(self.id_to_pairs[neg_id])
        sketch = self.transform(self._load_image(anchor["sketch"]))
        photo = self.transform(self._load_image(partner["photo"]))
        return sketch, photo, label


class RetrievalDataset(Dataset):
    def __init__(self, pairs, mode="photo"):
        assert mode in ("photo", "sketch")
        self.pairs = pairs
        self.mode = mode
        self.transform = get_transforms(train=False)
        all_ids = sorted({p["identity"] for p in pairs})
        self.id_to_label = {id_: i for i, id_ in enumerate(all_ids)}

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        img = Image.open(p[self.mode]).convert("RGB")
        img = self.transform(img)
        label = self.id_to_label[p["identity"]]
        return img, label


# --- Build loaders ---
BATCH_SIZE = 32

train_ds = SiamesePairDataset(splits["train"], train=True, seed=SEED)
val_ds = SiamesePairDataset(splits["val"], train=False, seed=SEED)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

# Sanity check: load one batch
s, p, l = next(iter(train_loader))
print(f"Batch shapes — sketch: {s.shape}, photo: {p.shape}, labels: {l.shape}")

## 4 — Model: Siamese Network with ResNet-50 Backbone

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=256, pretrained=True):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        backbone = models.resnet50(weights=weights)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.projection = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embedding_dim),
        )
        self._freeze_early_layers()

    def _freeze_early_layers(self):
        children = list(self.features.children())
        for child in children[:6]:
            for param in child.parameters():
                param.requires_grad = False

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.projection(x)
        x = F.normalize(x, p=2, dim=1)
        return x


class SiameseNetwork(nn.Module):
    def __init__(self, embedding_dim=256, pretrained=True):
        super().__init__()
        self.embedding_net = EmbeddingNet(embedding_dim, pretrained)

    def forward(self, sketch, photo):
        return self.embedding_net(sketch), self.embedding_net(photo)

    def get_embedding(self, x):
        return self.embedding_net(x)


class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        distance = F.pairwise_distance(emb1, emb2)
        label = label.float()
        pos_loss = label * distance.pow(2)
        neg_loss = (1 - label) * F.relu(self.margin - distance).pow(2)
        return (pos_loss + neg_loss).mean()


# --- Instantiate ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_DIM = 256

model = SiameseNetwork(embedding_dim=EMBEDDING_DIM, pretrained=True).to(device)
criterion = ContrastiveLoss(margin=1.0)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Device: {device}")
print(f"Trainable params: {trainable:,}")
print(f"Frozen params:    {frozen:,}")
print(f"Total params:     {trainable + frozen:,}")

## 5 — Training Loop

In [ ]:
import time
from tqdm.auto import tqdm

# ---- Hyperparameters ----
EPOCHS = 20
LR = 1e-4

optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-5,
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, n = 0.0, 0
    for sketches, photos, labels in tqdm(loader, desc="  Train", leave=False):
        sketches, photos, labels = sketches.to(device), photos.to(device), labels.to(device)
        emb_s, emb_p = model(sketches, photos)
        loss = criterion(emb_s, emb_p, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    for sketches, photos, labels in tqdm(loader, desc="  Val", leave=False):
        sketches, photos, labels = sketches.to(device), photos.to(device), labels.to(device)
        emb_s, emb_p = model(sketches, photos)
        loss = criterion(emb_s, emb_p, labels)
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)


# ---- Training ----
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

best_val_loss = float("inf")
history = []

print(f"Training for {EPOCHS} epochs on {device}\n")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)
    scheduler.step()
    elapsed = time.time() - t0

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": optimizer.param_groups[0]["lr"],
        "time_s": elapsed,
    }
    history.append(record)

    print(
        f"Epoch {epoch:3d}/{EPOCHS} | "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e} | {elapsed:.1f}s"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": best_val_loss,
            "args": {"embedding_dim": EMBEDDING_DIM},
        }, os.path.join(OUTPUT_DIR, "best_model.pth"))
        print(f"  ✅ Saved best model (val={best_val_loss:.4f})")

# Save final model & history
torch.save({"epoch": EPOCHS, "model_state_dict": model.state_dict(),
            "args": {"embedding_dim": EMBEDDING_DIM}},
           os.path.join(OUTPUT_DIR, "final_model.pth"))
with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)

print(f"\n🎉 Done! Best val loss: {best_val_loss:.4f}")

## 6 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_list = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_list, train_losses, "o-", label="Train Loss")
ax.plot(epochs_list, val_losses, "s-", label="Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Contrastive Loss")
ax.set_title("Training Curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## 7 — Evaluation: Retrieval Accuracy

In [ ]:
# Load best model
ckpt = torch.load(os.path.join(OUTPUT_DIR, "best_model.pth"), map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best model from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")

# Build retrieval datasets from test split
test_pairs = splits["test"]
print(f"Test set: {len(test_pairs)} pairs")

sketch_ds = RetrievalDataset(test_pairs, mode="sketch")
photo_ds = RetrievalDataset(test_pairs, mode="photo")

sketch_loader = torch.utils.data.DataLoader(sketch_ds, batch_size=32, shuffle=False, num_workers=2)
photo_loader = torch.utils.data.DataLoader(photo_ds, batch_size=32, shuffle=False, num_workers=2)


@torch.no_grad()
def compute_embeddings(model, loader, device):
    model.eval()
    all_emb, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="  Embedding", leave=False):
        emb = model.get_embedding(imgs.to(device))
        all_emb.append(emb.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_emb), np.concatenate(all_labels)


print("Computing embeddings...")
sketch_emb, sketch_labels = compute_embeddings(model, sketch_loader, device)
photo_emb, photo_labels = compute_embeddings(model, photo_loader, device)


def retrieval_accuracy(sketch_emb, sketch_labels, photo_emb, photo_labels, top_k=[1, 5, 10]):
    sim = sketch_emb @ photo_emb.T
    results = {}
    for k in top_k:
        correct = 0
        for i in range(len(sketch_labels)):
            top_indices = np.argsort(-sim[i])[:k]
            if sketch_labels[i] in photo_labels[top_indices]:
                correct += 1
        results[f"rank_{k}"] = correct / len(sketch_labels)
    return results


results = retrieval_accuracy(sketch_emb, sketch_labels, photo_emb, photo_labels)
print("\n📊 Retrieval Accuracy:")
for k, v in results.items():
    print(f"  {k}: {v:.4f} ({v * 100:.1f}%)")

# Save
with open(os.path.join(OUTPUT_DIR, "eval_results.json"), "w") as f:
    json.dump(results, f, indent=2)

## 8 — Retrieval Visualization

In [ ]:
def visualize_retrieval(test_pairs, sketch_emb, sketch_labels,
                        photo_emb, photo_labels, n_queries=5, top_k=5):
    sim = sketch_emb @ photo_emb.T
    fig, axes = plt.subplots(n_queries, top_k + 1, figsize=(3 * (top_k + 1), 3 * n_queries))
    if n_queries == 1:
        axes = axes[np.newaxis, :]

    indices = np.linspace(0, len(test_pairs) - 1, n_queries, dtype=int)

    for row, idx in enumerate(indices):
        sketch_img = Image.open(test_pairs[idx]["sketch"]).convert("RGB")
        axes[row, 0].imshow(sketch_img)
        axes[row, 0].set_title("Query sketch", fontsize=8)
        axes[row, 0].axis("off")

        top_indices = np.argsort(-sim[idx])[:top_k]
        true_label = sketch_labels[idx]

        for col, photo_idx in enumerate(top_indices):
            if photo_idx >= len(test_pairs):
                continue
            photo_img = Image.open(test_pairs[photo_idx]["photo"]).convert("RGB")
            axes[row, col + 1].imshow(photo_img)
            is_match = photo_labels[photo_idx] == true_label
            color = "green" if is_match else "red"
            axes[row, col + 1].set_title(
                f"{'✓' if is_match else '✗'} sim={sim[idx, photo_idx]:.2f}",
                fontsize=8, color=color,
            )
            axes[row, col + 1].axis("off")

    plt.suptitle("Sketch → Photo Retrieval (green = correct match)", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "retrieval_visualization.png"), dpi=150, bbox_inches="tight")
    plt.show()


visualize_retrieval(test_pairs, sketch_emb, sketch_labels, photo_emb, photo_labels)

## 9 — Download Outputs

Download the trained model, results, and visualizations to your local machine.

In [ ]:
import shutil
from google.colab import files

# Zip outputs
shutil.make_archive("outputs_bundle", "zip", ".", OUTPUT_DIR)
print("Contents:")
for f in os.listdir(OUTPUT_DIR):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

files.download("outputs_bundle.zip")
print("\n✅ Download started!")